## **Практика: LIME**

В теоретическом блоке мы разобрали построение LIME на верхнем уровне. Если вдаваться в подробности, то под капотом LIME также делает (или может делать) интересные преобразования, которые способны усилить и специализировать объяснение. Они отражены в гиперпараметрах. Но вместо того, чтобы использовать оболочку LIME в этом практическом занятии мы разберемся со всем и построим две собственные суррогатные модели с нуля, воссоздав подход статьи [bLIMEy: Surrogate Prediction Explanations Beyond LIME](https://arxiv.org/abs/1910.13016) (Sokol и др., 2019).  

Приступим!

![temp-Image1mlt-QZ.avif](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/temp-Image1mlt-QZ.avif)



In [ ]:
!pip install lime "fat-forensics[all]" -q #установка необходимых библиотек

In [ ]:
import fatf
import lime
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

fatf.setup_random_seed(42)

Подготовим данные. Для простоты, будем работать с очень известным в статистике датасетом — Ирисы Фишера. Он представляет собой измерения, снятые со 150 экземпляров ириса, по 50 экземпляров каждого из трех видов — Ирис щетинистый (Iris setosa), Ирис виргинский (Iris virginica) и Ирис разноцветный (Iris versicolor).

Для них представлены по четыре характеристики (в сантиметрах):

- Длина наружной доли околоцветника (англ. sepal length);
- Ширина наружной доли околоцветника (англ. sepal width);
- Длина внутренней доли околоцветника (англ. petal length);
- Ширина внутренней доли околоцветника (англ. petal width).

На основании этого набора данных требуется построить правило классификации, определяющее вид растения по данным измерений. Это задача многоклассовой классификации, так как имеется три класса — три вида ириса.

In [ ]:
#Загрузка

iris_data_dict = load_iris()
iris_data = iris_data_dict['data']
iris_target = iris_data_dict['target']
iris_feature_names = iris_data_dict['feature_names']
iris_target_names = iris_data_dict['target_names']

X_train, X_test, y_train, y_test = train_test_split(iris_data, iris_target, random_state=42)

В качестве модели "черного ящика" будем исползовать случайный лес.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics

blackbox_model = RandomForestClassifier(random_state=42, max_depth=2)
blackbox_model.fit(X_train, y_train)

predictions = blackbox_model.predict(X_test)
acc = sklearn.metrics.accuracy_score(y_test, predictions)

print(f'Model accuracy: {acc}')

In [ ]:
data_point = X_train[37] #Выберем произвольную точку данных

data_point_probabilities = blackbox_model.predict_proba(data_point.reshape(1, -1))[0]
data_point_probabilities

In [ ]:
data_point

In [ ]:
data_point_prediction = data_point_probabilities.argmax(axis=0) #Смотрим на класс для точки данных

data_point_class = iris_target_names[data_point_prediction]
data_point_class

Отлично! Рассмотрим разноцветный ирис. Чтобы в прямом смысле посмотреть на него, визуализируем набор данных и выбранную точку данных.

In [ ]:
X_train[y_train==1][:, 2]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

_ = plt.title('Наблюдаемые и интерпретируемое значение')

## Build LIME model

Напомним некоторые сведения из теоретического урока. При обучении суррогатной модели $g(z)$ для модели $f(x)$ мы решаем задачу вида:

$$\xi(x) = argmin_{g \in G} L(f, g, \pi_x) + \Omega(g).$$

В функции потерь $L$ присутствует нестандартный параметр $\pi_x$. Он отвечает за корректировку вклада объектов из окресности, назначая им веса и в функции $L$ является некоторой поправкой при вычислении квадратичной ошибки.

$$L(f, g, \pi_x) = \pi_x(f(z) - g(x'))^2,$$

**Как рассчитывается вес каждого объекта $\pi_x$ в функции потерь, минимизируемой при обучении суррогатной модели?**

Напомним, формула: $L(f, g, \pi_x) = \sum_{z} \pi_x(z)\,(f(z) - g(z'))^2,$

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/1) · [шаг на Степике](https://stepik.org/lesson/1335312/step/2)

В оригинальной имплементации (то есть в коде к статье [“Why Should I Trust You?” Explaining the Predictions of Any Classifier](https://arxiv.org/pdf/1602.04938)) обучать локальную суррогатную модель можно с *дискретизацией* непрерывных признаков и без, а для некоторых модальностей данных (текст, изображения) используется бинаризация. Эти понятия нам еще незнакомы, поэтому разберем их.

In [ ]:
import fatf.utils.data.discretisation as fatf_discretisation #импортируем вспомогательные функции
import fatf.utils.data.augmentation as fatf_augmentation

Первый шаг алгоритма LIME генерация данных, похожих на исходные.

In [ ]:
fatf.setup_random_seed(42)  # сид ставим заново в этой ячейке: повторный запуск дает ту же выборку
augmenter = fatf_augmentation.Mixup(X_train, ground_truth=y_train)

sampled_data = augmenter.sample(data_point, samples_number=100) #Сгенерировали точки на основе исходных
sampled_data_probabilities = blackbox_model.predict_proba(sampled_data) #Спрогнозировали вероятности для сгенерированных точек

Посмотрим на новые точки.

In [ ]:
sampled_data_predictions = sampled_data_probabilities.argmax(axis=1)
sampled_data_0_indices = np.where(sampled_data_predictions == 0)[0]
sampled_data_1_indices = np.where(sampled_data_predictions == 1)[0]
sampled_data_2_indices = np.where(sampled_data_predictions == 2)[0]

_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')


_ = plt.scatter(
     sampled_data[sampled_data_0_indices, 2],
     sampled_data[sampled_data_0_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[0]))
_ = plt.scatter(
     sampled_data[sampled_data_1_indices, 2],
     sampled_data[sampled_data_1_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[1]))
_ = plt.scatter(
     sampled_data[sampled_data_2_indices, 2],
     sampled_data[sampled_data_2_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[2]))

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

Отметим, что синтетические точки **похожи** на исходные и находятся близко ко всем классам. Однако это не единственный подход к созданию синтетических данных для прогонза. Новые данные также можно генерировать **только в окресности** объясняемой точки. За это в официальной реализации Lime отвечает гиперпараметр `sample_around_instance`.


Мы же реализуем подход, когда генерация синтетических точек делается на основе **всех** данных.

Далее двинемся к ключевым изучаемым вещаем: процесса дискретизации и бинаризации данных. \

**Дискретизация** — процесс преобразования некоторой непрерывной функции к диксретной. В данном случае (и это является одной из возможных реализаций алгоритма LIME из коробки), дискретизация происходит при помощи разбиения на *квартили*.


**Квартили** — это значения, которые делят упорядоченный набор данных на четыре равных части, каждая из которых содержит 25% данных. Процесс дискретизации (разбиения) на квартили называется квартильной дискретизацией.
Она происходит следующим образом:
1. Извлекаем границы каждого квартиля
2. Каждую точку в наборе данных кодируем как вектор с координатами из мноества {0, 1, 2, 3} по правилу — заменяем значение координаты на номер квартиля (0, 1, 2, 3), в который она попадает.

**Какие координаты у вектора $[2.3, 5.1, 2., 3.3]$ получатся после дискретизации, если границы квартилей такие: $[0, 2.1], [2.2, 3], [3.1, 4], [4.1, 5.3]$**

В ответ запишите координату номер 2 (координаты нумеруются с единицы, интервалы — с нуля, как в `lime`).

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/2) · [шаг на Степике](https://stepik.org/lesson/1335312/step/3)

In [ ]:
discretiser = fatf_discretisation.QuartileDiscretiser(
    X_train,
    feature_names=iris_feature_names)

data_point_discretised = discretiser.discretise(data_point)
sampled_data_discretised = discretiser.discretise(sampled_data)

discretiser.feature_bin_boundaries

Убедимся, что значения границ — действительно квартили.

In [ ]:
import pandas as pd

pd.DataFrame(X_train, columns=iris_feature_names).describe()[3:] #Дествительно, границы совпадают с нужными нам статистиками.

In [ ]:
#Посмотрим на дискретизированную точку
data_point_discretised

Теперь, чтобы получить *интерпретируемую модель* осуществим **бинаризацию данных**.
Бинаризация данных — как следут из названия — приведение всех координат к сочетанию нулей и единиц (бинарному виду). Бинаризовать будем по следующему правилу: ставить 0, если координаты дискретизированной точки данных не совпадают с точкой данных из сэмпла, и 1 иначе.

In [ ]:
import fatf.utils.data.transformation as fatf_transformation

sampled_data_binarised = fatf_transformation.dataset_row_masking(
    sampled_data_discretised, data_point_discretised)

fatf_transformation.dataset_row_masking(data_point_discretised.reshape(1, -1), data_point_discretised)

На практике бинаризация осуществляется для текстовых данных и изображений. На нашем примере она даст результаты, которые будут рассогласованы с другими подходами к интерпретации. Какие лучше — необходимо проверять на практике.

Еще раз рассмотрим все 3 трансформации:

In [ ]:
print(f'Исходная точка данных: {data_point}')
print(f'Дискретизированная точка данных: {data_point_discretised}')

print(f'Некоторый экземпляр из дискретизированных точек до бинаризации: {sampled_data_discretised[7]}')
print(f'Некоторый экземпляр из дискретизированных точек после бинаризации: {sampled_data_binarised[7]}')

В качестве последнего шага осталость вычислить веса объектов для функции ошибки. Это  мы просто сделаем по соответствующей формуле, которую вы вспоминали на задаче выше.

In [ ]:
import fatf.utils.distances as fatf_distances
import fatf.utils.kernels as fatf_kernels

features_number = sampled_data_binarised.shape[1]
kernel_width = np.sqrt(features_number) * 0.75

distances = fatf_distances.euclidean_point_distance(np.ones(features_number), sampled_data_binarised) #расстояния вычислим на основе бинаризованных данных
weights = fatf_kernels.exponential_kernel(
     distances, width=kernel_width)

## **Обучение локального алгоритма**

Задача, решаемая моделями для датасета с ирисами Фишера, является задачей мультиклассовой классификации. Таким образом, тренируя суррогатную модель мы можем обучить ее двумя способами:
- используя подход ONE VS REST, где суррогатная модель будет прогнозировать 0 или 1 в зависимости от факта принадлежности объекта к интересующему нас классу.
- используя классический подход, где суррогатная модель будет прогнозировать вектор вероятностей.

Преимущество первого подхода — концентрация на интересующем нас классе, второго — на универсальности — полученного суррогата можно использовать для объяснения объекта из любых классов.

Кроме того, выше мы отметили, что модель может быть обучена на бинаризованных данных и нет. Обучая на бинаризованных данных, получается модель, отвечающая на вопрос:

*«Если бы это конкретное значение признака объясняемой точки данных находилось за пределами диапазона (для числовых признаков) или имело другое значение (для категориального признака), как бы это повлияло на вероятность принадлежности этой точки к объясняемому классу (вероятностная классификация) / прогнозируемое числовое значение (регрессия)?»*

Это полезно для изображений, так как позволяет сранивать знаимость различных сегментов картинки.

В нашем же случае модель решим задачу минимизации прогнозируемых значений от истинных в классическом смысле.

In [ ]:
sampled_data_predictions_versicolor = sampled_data_probabilities[:, 1] # сохраним вероятности для подхода OVR

Инициализируем модель и обучим ее на дискретизированных данных на всех вероятностях.

In [ ]:
import sklearn.linear_model
import numpy as np

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_discretised, sampled_data_predictions, sample_weight=weights)
for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

**И вот вы обучили первую LIME!**
Зафиксируем важные признаки по порядку: `petal width`, `petal length`

Посмотрим, на реализованный вариант на не дискретизированных данных

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data, sampled_data_predictions, sample_weight=weights)  #посмотрим, на реализованный вариант на не дискретизированных данных

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

Зафиксируем важные признаки по порядку здесь: `petal width`, `petal length`

**Суррогат в подходе One vs Rest.** Выше суррогат обучался в классическом подходе — на прогнозах черного ящика по всем классам (`sampled_data_predictions`). Теперь обучим тот же Ridge в подходе One vs Rest: целевая переменная — вероятность класса versicolor `sampled_data_predictions_versicolor`, которую мы сохранили выше, веса — те же `weights`. Этот OVR-суррогат нужен в заданиях Q1, Q2 и Q4. В остальных заданиях ручная реализация — первая LIME выше, в классическом подходе.

**Q1. Обучите суррогат в подходе One vs Rest (целевая переменная — вероятность класса versicolor, `sampled_data_predictions_versicolor`) сначала на дискретизированных данных, потом на недискретизированных (те же веса). Меняются ли топ-2 признака по важности?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/3) · [шаг на Степике](https://stepik.org/lesson/1335312/step/4)

In [ ]:
# Ваш код здесь
# OVR-суррогат: sklearn.linear_model.Ridge(alpha=1, fit_intercept=True),
# целевая переменная sampled_data_predictions_versicolor, веса weights;
# обучите его на sampled_data_discretised, потом на sampled_data и сравните топ-2 признака по модулю coef_

**Q2.** Теперь поменяйте способ считать расстояния — на основе исходных (не дискретизированных) значений, но саму модель по-прежнему обучите на дискретизированных данных (тот же суррогат One vs Rest, что в Q1). Меняются ли топ-2 признака сильнее, чем от самой дискретизации в Q1?

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/4) · [шаг на Степике](https://stepik.org/lesson/1335312/step/5)

In [ ]:
# Ваш код здесь
# расстояния от data_point до sampled_data: fatf_distances.euclidean_point_distance,
# веса: fatf_kernels.exponential_kernel(..., width=kernel_width);
# OVR-суррогат из Q1 на sampled_data_discretised с новыми весами

**Получите интерпретацию при помощи ручной реализации на бинаризованных данных на весах `weights` (классический подход: прогнозы черного ящика по классам, `sampled_data_predictions`). Какие признаки (признак) подсвечиваются в этом случае?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/5) · [шаг на Степике](https://stepik.org/lesson/1335312/step/6)

In [ ]:
# Ваш код здесь
# ручная реализация, как первая LIME выше: Ridge(alpha=1, fit_intercept=True) с целевой переменной sampled_data_predictions,
# но на sampled_data_binarised и с весами weights

Числовой пример из теории: объясняем прогноз черного ящика $f$ в точке $x=0$ по пяти возмущениям $z$ с прогнозами $f(z)$. Веса $\pi_x(z)=\exp(-z^2/\sigma^2)$, наклон взвешенной линейной модели $\beta=\frac{\sum_z \pi_x(z)\,z\,f(z)}{\sum_z \pi_x(z)\,z^2}$. Ядро в `fatf` и `lime` берет из этой экспоненты еще и квадратный корень, здесь корня нет. Веса считаем точно, без округления до сотых, поэтому число может немного отличаться от разбора в теории.

**Q3. Воспроизведите числовой пример теории (1 признак, 5 точек, веса по формуле БЕЗ дополнительного корня). При узком ядре ($\sigma=1$) чему равен $\beta$? Округлите до сотых.**

Веса считаем точно ($e^{-4}$ и $e^{-1}$), поэтому число чуть отличается от того, что вышло в теории по округленным весам.

Впишите ответ в тренажер.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/6) · [шаг на Степике](https://stepik.org/lesson/1335312/step/7)

In [ ]:
z = np.array([-2., -1., 0., 1., 2.])    # возмущения вокруг x = 0
f_z = np.array([3., 0., 0.5, 1., 0.])  # прогнозы черного ящика f(z)
sigma = 1.                              # узкое ядро

# Ваш код здесь: веса pi_x(z) по формуле без корня и наклон beta

**Q4. Постройте суррогат при разных `kernel_width` (0.25x...4x от дефолта $0.75\sqrt d$) на реальном пайплайне (тот же объект, что и выше; суррогат в подходе One vs Rest — на вероятностях класса versicolor, `sampled_data_predictions_versicolor`). Какой признак лидирует по модулю коэффициента при УЗКОМ ядре, а какой — при ШИРОКОМ?**

Объект — тот же `data_point`, суррогат — OVR-суррогат из Q1 на дискретизированных данных, расстояния — те же `distances` по бинаризованным данным. Ширина ядра — параметр `width` в `fatf_kernels.exponential_kernel`, по умолчанию выше стоит `kernel_width` $=0.75\sqrt d$.

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/7) · [шаг на Степике](https://stepik.org/lesson/1335312/step/8)

In [ ]:
width_factors = [0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4]  # множители к kernel_width = 0.75 * sqrt(d)
# 0.1x в сетке нет: при нем все веса почти нулевые, коэффициенты ~1e-10, и лидера выбирает численный шум

# Ваш код здесь
# для каждого множителя: веса fatf_kernels.exponential_kernel(distances, width=factor * kernel_width),
# OVR-суррогат из Q1 на sampled_data_discretised и признак с наибольшим модулем коэффициента

**Библиотечная реализация**. \
Посмотрим, что дает библиотечная реализация. Все гиперпараметрв достаточно интуитивны, но на всякий случай поясним каждый:

- `class_names` — названия прогнозируемых классов
- `feature_names` — названия принаков
- `kernel_width`— ширина ядра, по умолчанию (и в нашем случае) $\sqrt{n\_features}*0.75$
- `verbose` — подробные выкладки при обучении суррогатной модели
- `discretizer` — подход к дискретизации
- `mode` — задача, реашемая моделью
- `discretize_continuous` — необходимо ли дискретизировать признаки
- `sample_around_instance` — генерировать ли синтетические данные **только в окресности** рассматриваемой точки

Последовательно посмотрим на важные признаки на дискретизированных и нет данных. Начнем с недискритиpированных.  

In [ ]:
from lime.lime_tabular import LimeTabularExplainer #посмотрим, что дает библиотечная реализация

explainer = LimeTabularExplainer(X_train,
                                 class_names=iris_target_names,
                                 feature_names=iris_feature_names,
                                 kernel_width=np.sqrt(features_number) * 0.75,
                                 verbose=False,
                                 mode='classification',
                                 discretize_continuous=False,
                                 sample_around_instance=False,
                                 random_state=42)

exp = explainer.explain_instance(data_point, blackbox_model.predict_proba, top_labels=1)
explained_label = exp.available_labels()[0]  # объясняемый класс
print(exp.as_list(label=explained_label))
_ = exp.as_pyplot_figure(label=explained_label)

Видим, что без дискретизации не видно явной силы влияния конкретных признаков.

**Q5. Получите интерпретацию с квартильной дискретизацией при помощи библиотеки LIME. Согласуется ли она с полученной выше при ручной реализации в классическом подходе?**

Ручная реализация здесь — первая LIME из раздела «Build LIME model»: классический подход, дискретизированные данные.

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/8) · [шаг на Степике](https://stepik.org/lesson/1335312/step/9)

In [ ]:
# Ваш код здесь
# LimeTabularExplainer как выше, но discretize_continuous=True и discretizer='quartile'

Видим, что результаты не равны друг другу в точноти, но похожи по общим выводам.

## Hints to tuning LIME:

1. Первое, что можно поменять в LIME, чтобы получить иное объяснение - окрестность точки или ширину ядра.

2. Второй подход к тюнингу — способы расчета значимости расстояния объектов до точки.

3. Ширина ядра окрестности также позволяет осуществлять тюнинг объяснения, однако наиболее часто используется ширина по умолчанию.

4. Дискретизация и бинаризация также позволяют более разносторонне использовать один алгоритм объяснения, получая ответы на другие формулировки вопроса к объекту.

## **Суррогатное дерево**

Линейная модель — не единственная в нашем арсенале. В некоторых случаях более полезным и информативным оказывается построение суррогатного дерева. Простой пример построения:

In [ ]:
import sklearn.tree

blimey_tree = sklearn.tree.DecisionTreeClassifier(max_depth=5, random_state=42)
blimey_tree.fit(sampled_data, sampled_data_predictions, sample_weight=weights)


In [ ]:
for n_i in zip(iris_feature_names, blimey_tree.feature_importances_):
     name, importance = n_i
     print('->{}<-: {}'.format(name, importance))


Также для интерпретации здесь полезно визуализировать правила разбиений.

In [ ]:
from sklearn import tree
print(tree.export_text(blimey_tree, feature_names=iris_feature_names))

Или более красиво.

In [ ]:
plt.figure(figsize=(18, 8))
_ = tree.plot_tree(blimey_tree,
                   feature_names=iris_feature_names,
                   class_names=iris_target_names,
                   filled=True)
plt.show()

**Q6. Проверьте `feature_importances_` построенного суррогатного дерева (`DecisionTreeClassifier(max_depth=5, random_state=42)`). Какой признак имеет важность ≈0, хотя формально встречается в тексте дерева как условие разбиения?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/10) · [шаг на Степике](https://stepik.org/lesson/1335312/step/11)

### **Выводы**
- LIME допускает разные подходы к формированию и оценке важности признаков
- Объяснения внутри различных сценариев LIME могут различаться, так что истинность необходимо проверять эмпирически, а гипотезы генерировать на основе комбинации методов/моделей
- Использование LIME предполагает, что используются изначально концептуально понятные признаки (кусочки изображения, значения числового вектора, соотносящиеся с реальными данными)

## **Глобальные обобщения LIME**

В теории разобраны два глобальных обобщения LIME: SP-LIME собирает глобальную картину из локальных объяснений, G-LIME вносит глобальную важность признаков в локальную подгонку. Ниже — по заданию на каждое.

**Q7. Примените `SubmodularPick` (SP-LIME) из библиотеки `lime` к тестовой выборке, чтобы выбрать репрезентативный набор объяснений (`num_exps_desired`). Сколько объектов было отобрано?**

Впишите ответ в тренажер.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/12) · [шаг на Степике](https://stepik.org/lesson/1335312/step/13)

In [ ]:
from lime import submodular_pick

num_exps_desired = 5  # сколько объяснений просим отобрать

explainer_sp = LimeTabularExplainer(X_train,
                                    class_names=iris_target_names,
                                    feature_names=iris_feature_names,
                                    discretize_continuous=True,
                                    discretizer='quartile',
                                    mode='classification',
                                    random_state=42)

# Ваш код здесь
# submodular_pick.SubmodularPick(explainer_sp, X_test, blackbox_model.predict_proba,
#                                method='full', num_features=4, num_exps_desired=num_exps_desired);
# отобранные объяснения лежат в атрибуте sp_explanations, номера объектов — в атрибуте V

**Q8 (бонус). G-LIME добавляет к локальной задаче LIME глобальный регуляризатор, стягивающий коэффициенты $\beta$ к глобальной оценке важности $\beta_0$ (например, из SHAP или предобученной глобальной модели), с силой $\lambda_2$. Что происходит с $\|\beta-\beta_0\|$ при увеличении $\lambda_2$ от 0 до больших значений?**

Выберите ответ в тренажере.

Ответ — в тренажере: [шаг на сайте](https://ai-interpretability.school/learn/7/7/13) · [шаг на Степике](https://stepik.org/lesson/1335312/step/14)

In [ ]:
lambdas_2 = [0, 0.1, 1, 10, 100, 1000]

# глобальный приор beta_0: линейная модель по всей обучающей выборке
# (дискретизированные признаки, вероятность класса versicolor от черного ящика)
X_train_discretised = discretiser.discretise(X_train)
global_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)
global_model.fit(X_train_discretised, blackbox_model.predict_proba(X_train)[:, 1])
beta_0 = global_model.coef_

# локальная задача — OVR-суррогат из Q1
Z = sampled_data_discretised.astype(float)
y = sampled_data_predictions_versicolor
w = weights

# Ваш код здесь
# для каждого lambda_2 (lambda_1 = 0) решите
#   min_beta  sum_z w (y - b - beta^T z)^2 + lambda_2 ||beta - beta_0||^2
# в закрытой форме: центрируйте Z и y взвешенными средними, тогда
#   beta = (Zc^T W Zc + lambda_2 I)^{-1} (Zc^T W yc + lambda_2 beta_0),
# и выведите ||beta - beta_0||